# The decision tree algorithm
In this exercise we will build a decision tree model and a random forest model, which will assign the iris class based on the measured values.

Again, we will build on the prepared data from the last Jupyter notebook.

## Loading and splitting the data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
my_arrays = np.load("iris_numpy.npz")
X = my_arrays['X']
y = my_arrays['y']
X_norm = my_arrays['X_norm']
X_features = my_arrays['X_feature']

Splitting the data into training and test sets

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2, random_state=42)

We will load the encoder and display the mapping of class names to numbers.

In [ ]:
import joblib
encoder = joblib.load('classification_encoder.bin')
encoder_mapping = dict(enumerate(encoder.classes_))
encoder_mapping

# Decision tree

A decision tree is a supervised machine learning algorithm. The model builds a set of conditions that lead it to the final prediction. The result of training is a tree structure — each node contains a condition on one variable and leads either to another node or to a leaf with a prediction.

**Advantages:**
- Simple to visualize and interpret
- Doesn't need data normalization (we use it for consistency with the other models)
- Handles both numeric and categorical variables

**Disadvantages:**
- Prone to overfitting
- Unstable — small changes in the data can lead to a completely different tree

## Training

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

Visualization of the trained tree. Each node shows the splitting condition, the Gini value (a measure of the node's impurity), the number of samples, and the class distribution.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 8))
plot_tree(
    dt_model,
    feature_names=['petal_length', 'petal_width'],
    class_names=encoder.classes_,
    filled=True,
    rounded=True,
    fontsize=10,
)
plt.title('Decision Tree')
plt.tight_layout()
plt.show()

## Prediction
Running the model on the training and test data.

In [ ]:
y_pred_train = dt_model.predict(X_train)
y_pred_test = dt_model.predict(X_test)

## Model visualization
We will create two charts. One will show the predictions, the other the actual values.

Charts for the training data

In [ ]:
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
for enc, name in enumerate(encoder.classes_):
    mask = y_pred_train == enc
    plt.scatter(X_train[mask, 0], X_train[mask, 1], s=50, label=name)
plt.title('Predicted Species (train)')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.legend()

plt.subplot(1, 2, 2)
for enc, name in enumerate(encoder.classes_):
    mask = y_train == enc
    plt.scatter(X_train[mask, 0], X_train[mask, 1], s=50, label=name)
plt.title('True Species (train)')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.legend()
plt.tight_layout()
plt.show()

Charts for the test data

In [ ]:
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
for enc, name in enumerate(encoder.classes_):
    mask = y_pred_test == enc
    plt.scatter(X_test[mask, 0], X_test[mask, 1], s=50, label=name)
plt.title('Predicted Species (test)')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.legend()

plt.subplot(1, 2, 2)
for enc, name in enumerate(encoder.classes_):
    mask = y_test == enc
    plt.scatter(X_test[mask, 0], X_test[mask, 1], s=50, label=name)
plt.title('True Species (test)')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.legend()
plt.tight_layout()
plt.show()

## Model evaluation
We can tell how well the model works from various metrics. The task's requirements will suggest which metric to choose.

| Metric | Formula | Description |
|---------|--------|-------|
| **Accuracy** | (TP + TN) / (TP + TN + FP + FN) | Proportion of correct answers |
| **Recall** (Sensitivity, TPR) | TP / (TP + FN) | Of the positives, how many we correctly caught |
| **Specificity** (TNR) | TN / (TN + FP) | Of the negatives, how many we correctly rejected |
| **Precision** | TP / (TP + FP) | Of those labeled as positive, how many actually are |
| **F1** | 2 × (Precision × Recall) / (Precision + Recall) | Harmonic mean of precision and recall |

Abbreviations: TP = True Positive, TN = True Negative, FP = False Positive, FN = False Negative

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import seaborn as sns

The confusion matrix shows how the model predicts for each class. The correct predictions are on the main diagonal. The other cells show the mix-ups.

In [ ]:
cf_matrix = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cf_matrix, annot=True, xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — Decision Tree')
plt.show()

The model's score — accuracy and a classification report with precision, recall, and F1 for each class.

In [ ]:
print(f'Train accuracy: {accuracy_score(y_train, y_pred_train):.4f}')
print(f'Test accuracy:  {accuracy_score(y_test, y_pred_test):.4f}')
print()
print(classification_report(y_test, y_pred_test, target_names=encoder.classes_))

## Hyperparameter tuning
A decision tree has several important parameters:
- `max_depth` — the maximum depth of the tree; a smaller value prevents overfitting
- `criterion` — the splitting criterion: `gini` (impurity measure) or `entropy` (information gain)
- `min_samples_split` — the minimum number of samples required to split a node

GridSearchCV automatically tries all combinations and selects the best one.

In [ ]:
from sklearn.model_selection import GridSearchCV

dt_params = {
    'max_depth': [2, 3, 4, 5, None],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10],
}

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5, scoring='accuracy')
grid_dt.fit(X_train, y_train)

print(f'Best parameters: {grid_dt.best_params_}')
print(f'Best CV accuracy: {grid_dt.best_score_:.4f}')

## Saving the model
The model can again be saved to a file for use in inference.

In [ ]:
joblib.dump(dt_model, 'dt_model.bin')

## Using the model

We will try to use the model to determine 3 different irises. The input values are given in the original units, the same as the original data.

In [ ]:
iris_measurements = np.array([[5.1, 1.9], [1.0, 0.5], [6.0, 2.5]])

The model was trained on normalized data, so we need to normalize the input data.

In [ ]:
scaler = joblib.load('classification_std_scaler.bin')
iris_measurements_norm = scaler.transform(iris_measurements)

loaded_dt = joblib.load('dt_model.bin')
predictions = loaded_dt.predict(iris_measurements_norm)

for pred in predictions:
    print(f'Predicted class: {encoder_mapping[pred]}')

# Random forest

A random forest is an extension of the decision tree. Instead of a single tree, it builds a large number of trees (a forest), each trained on a different random subset of the data and variables (bagging). The final prediction is produced by voting across all the trees — the class with the most votes wins.

**Advantages:**
- More resistant to overfitting than a single tree
- More stable predictions
- Naturally provides feature importance

**Disadvantages:**
- Harder to interpret than a single tree
- Slower training and inference

## Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

## Prediction
Running the model on the training and test data.

In [ ]:
y_rf_pred_train = rf_model.predict(X_train)
y_rf_pred_test = rf_model.predict(X_test)

## Model visualization

Feature importance expresses how much each variable contributes to the forest's decisions. It is calculated as the average decrease in Gini impurity across all the trees.

In [ ]:
import pandas as pd

feature_importances = pd.Series(rf_model.feature_importances_, index=['petal_length', 'petal_width'])
feature_importances.sort_values().plot(kind='barh', figsize=(6, 3), color='steelblue')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

Charts for the training data

In [ ]:
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
for enc, name in enumerate(encoder.classes_):
    mask = y_rf_pred_train == enc
    plt.scatter(X_train[mask, 0], X_train[mask, 1], s=50, label=name)
plt.title('Predicted Species (train)')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.legend()

plt.subplot(1, 2, 2)
for enc, name in enumerate(encoder.classes_):
    mask = y_train == enc
    plt.scatter(X_train[mask, 0], X_train[mask, 1], s=50, label=name)
plt.title('True Species (train)')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.legend()
plt.tight_layout()
plt.show()

Charts for the test data

In [ ]:
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
for enc, name in enumerate(encoder.classes_):
    mask = y_rf_pred_test == enc
    plt.scatter(X_test[mask, 0], X_test[mask, 1], s=50, label=name)
plt.title('Predicted Species (test)')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.legend()

plt.subplot(1, 2, 2)
for enc, name in enumerate(encoder.classes_):
    mask = y_test == enc
    plt.scatter(X_test[mask, 0], X_test[mask, 1], s=50, label=name)
plt.title('True Species (test)')
plt.xlabel('petal_length')
plt.ylabel('petal_width')
plt.legend()
plt.tight_layout()
plt.show()

## Model evaluation

The confusion matrix shows how the model predicts for each class. The correct predictions are on the main diagonal. The other cells show the mix-ups.

In [ ]:
cf_matrix_rf = confusion_matrix(y_test, y_rf_pred_test)
sns.heatmap(cf_matrix_rf, annot=True, xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — Random Forest')
plt.show()

The model's score — accuracy and a classification report with precision, recall, and F1 for each class.

In [ ]:
print(f'Train accuracy: {accuracy_score(y_train, y_rf_pred_train):.4f}')
print(f'Test accuracy:  {accuracy_score(y_test, y_rf_pred_test):.4f}')
print()
print(classification_report(y_test, y_rf_pred_test, target_names=encoder.classes_))

## Hyperparameter tuning
Random forest has these key parameters:
- `n_estimators` — the number of trees in the forest; more trees = more stable predictions, slower training
- `max_depth` — the maximum depth of each tree
- `min_samples_split` — the minimum number of samples required to split a node

In [ ]:
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5],
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy')
grid_rf.fit(X_train, y_train)

print(f'Best parameters: {grid_rf.best_params_}')
print(f'Best CV accuracy: {grid_rf.best_score_:.4f}')

## Saving the model
The model can again be saved to a file for use in inference.

In [ ]:
joblib.dump(rf_model, 'rf_model.bin')

## Using the model

In [ ]:
loaded_rf = joblib.load('rf_model.bin')
predictions_rf = loaded_rf.predict(iris_measurements_norm)

for pred in predictions_rf:
    print(f'Predicted class: {encoder_mapping[pred]}')

## Comparing the models
Comparing the accuracy of the decision tree and the random forest on the training and test data.

In [ ]:
results = pd.DataFrame([
    {'Model': 'Decision Tree',
     'Train Accuracy': accuracy_score(y_train, y_pred_train),
     'Test Accuracy': accuracy_score(y_test, y_pred_test)},
    {'Model': 'Random Forest',
     'Train Accuracy': accuracy_score(y_train, y_rf_pred_train),
     'Test Accuracy': accuracy_score(y_test, y_rf_pred_test)},
])
results.round(4)